In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ユニタリ演算の合成

<details>
<summary><b>パッケージのバージョン</b></summary>

このページのコードは、以下の要件を用いて開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
```
</details>

ユニタリ演算は、量子系に対するノルム保存の変換を記述します。
$n$ 量子ビットの場合、この変換は $2^n \times 2^n$ の複素行列 $U$ によって表され、その随伴行列が逆行列と等しい、すなわち $U^\dagger U = \mathbb{1}$ という条件を満たします。

特定のユニタリ演算を量子ゲートの集合へ合成することは、例えば量子アルゴリズムの設計・応用や量子回路のコンパイルにおいて用いられる基本的なタスクです。

Clifford ゲートで構成されるユニタリやテンソル積構造を持つユニタリなど、特定のクラスに対しては効率的な合成が可能ですが、ほとんどのユニタリはこれらのカテゴリに該当しません。
一般のユニタリ行列に対する合成は複雑なタスクであり、計算コストは量子ビット数に対して指数的に増大します。
そのため、実装したいユニタリに対して効率的な分解が既知であれば、一般的な合成よりもそちらを使用する方が適切です。

> **Note:** 分解が利用できない場合、Qiskit SDK は分解を見つけるためのツールを提供しています。
>     ただし、一般にこれは深い回路を生成するため、ノイズのある量子コンピュータでの実行には適さない場合があることに注意してください。

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## 回路最適化のための再合成
長い1量子ビットおよび2量子ビットゲートの列を再合成することで、回路の長さを短縮できる場合があります。例えば、次の回路は3つの2量子ビットゲートを使用しています。

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Output of the previous code cell](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

しかし、以下のコードで再合成を行うと、必要な CX ゲートは1つだけになります。（ここでは、ユニタリの再合成に使用されるゲートをより明確に可視化するために `QuantumCircuit.decompose()` メソッドを使用しています。）